In [1]:
"""
CS Master's Program – Cybersecurity
Week 4 Homework: Diffie-Hellman Key Exchange
Student: Haseeb Ali Syed

This script demonstrates:
  - Section 2: Manual DH Key Exchange between Alice and Bob
  - Section 3: Man-in-the-Middle (MITM) Attack by Eve
"""

# ──────────────────────────────────────────────────────────────
# HELPER FUNCTION
# ──────────────────────────────────────────────────────────────

def mod_exp(base, exponent, modulus):
    """
    Computes:  base^exponent mod modulus
    Python's built-in pow(base, exp, mod) does this efficiently,
    but we spell it out here so you can see the formula clearly.
    """
    return pow(base, exponent, modulus)


# ──────────────────────────────────────────────────────────────
# SECTION 2: NORMAL DH KEY EXCHANGE
# Parameters given in the problem set
# ──────────────────────────────────────────────────────────────

print("=" * 55)
print("  SECTION 2: Diffie-Hellman Key Exchange")
print("=" * 55)

# Public parameters (known to everyone, including attackers)
p = 23   # Prime modulus
g = 5    # Generator

# Private secrets (chosen secretly by each party)
a = 4    # Alice's private key
b = 3    # Bob's private key

print(f"\n[Public Parameters]")
print(f"  Prime modulus  p = {p}")
print(f"  Generator      g = {g}")

print(f"\n[Private Secrets]")
print(f"  Alice's secret a = {a}")
print(f"  Bob's secret   b = {b}")

# Step (a): Alice computes her public value
# Formula: A = g^a mod p
A = mod_exp(g, a, p)   # 5^4 mod 23 = 625 mod 23 = 4
print(f"\n[Step a] Alice computes her public value:")
print(f"  A = g^a mod p = {g}^{a} mod {p} = {A}")

# Step (b): Bob computes his public value
# Formula: B = g^b mod p
B = mod_exp(g, b, p)   # 5^3 mod 23 = 125 mod 23 = 10
print(f"\n[Step b] Bob computes his public value:")
print(f"  B = g^b mod p = {g}^{b} mod {p} = {B}")

# Alice and Bob exchange their public values over the network.
# An eavesdropper can see A and B, but NOT a or b.

# Step (c): Alice derives the shared secret using Bob's public value
# Formula: s = B^a mod p
alice_secret = mod_exp(B, a, p)   # 10^4 mod 23 = 18
print(f"\n[Step c] Alice computes the shared secret:")
print(f"  s = B^a mod p = {B}^{a} mod {p} = {alice_secret}")

# Step (d): Bob derives the shared secret using Alice's public value
# Formula: s = A^b mod p
bob_secret = mod_exp(A, b, p)     # 4^3 mod 23 = 18
print(f"\n[Step d] Bob computes the shared secret:")
print(f"  s = A^b mod p = {A}^{b} mod {p} = {bob_secret}")

# Step (e): Verify both arrived at the same secret
print(f"\n[Step e] Verification:")
if alice_secret == bob_secret:
    print(f"  ✓ SUCCESS! Both computed the same shared secret: {alice_secret}")
else:
    print(f"  ✗ MISMATCH! Alice={alice_secret}, Bob={bob_secret}")


# ──────────────────────────────────────────────────────────────
# SECTION 3: MAN-IN-THE-MIDDLE (MITM) ATTACK BY EVE
# ──────────────────────────────────────────────────────────────

print("\n")
print("=" * 55)
print("  SECTION 3: Man-in-the-Middle Attack (Eve)")
print("=" * 55)

# Eve's private secret (chosen by the attacker)
e = 6    # Eve's private key

# Eve computes her own public value to impersonate both parties
# Formula: E = g^e mod p
E = mod_exp(g, e, p)   # 5^6 mod 23 = 15625 mod 23 = 8
print(f"\n[Setup] Eve's private key: e = {e}")
print(f"  Eve's public value: E = g^e mod p = {g}^{e} mod {p} = {E}")

# ── Step (a): Alice → Bob (Eve intercepts) ──
print(f"\n[Step a] Alice sends A={A} to Bob.")
print(f"  Eve INTERCEPTS A={A} and sends her own value E={E} to Bob instead.")
print(f"  Bob now thinks he received Alice's key, but it's Eve's.")

# ── Step (b): Bob → Alice (Eve intercepts) ──
print(f"\n[Step b] Bob sends B={B} to Alice.")
print(f"  Eve INTERCEPTS B={B} and sends her own value E={E} to Alice instead.")
print(f"  Alice now thinks she received Bob's key, but it's Eve's.")

# ── Step (c): Key Establishment ──
print(f"\n[Step c] Key Establishment:")

# Alice-Eve shared key:
# Alice uses the value she received (E=8) and her own private key (a=4)
# Formula: K_AE = E^a mod p
K_AE = mod_exp(E, a, p)   # 8^4 mod 23 = 4096 mod 23 = 2
print(f"\n  Alice-Eve Key (K_AE):")
print(f"    Alice computes: K_AE = E^a mod p = {E}^{a} mod {p} = {K_AE}")

# Eve computes the same Alice-Eve key:
# Eve knows her own secret (e=6) and Alice's real public key (A=4)
# Formula: K_AE = A^e mod p
eve_to_alice = mod_exp(A, e, p)   # 4^6 mod 23 = 4096 mod 23 = 2
print(f"    Eve computes:   K_AE = A^e mod p = {A}^{e} mod {p} = {eve_to_alice}")

# Bob-Eve shared key:
# Bob uses the value he received (E=8) and his own private key (b=3)
# Formula: K_EB = E^b mod p
K_EB = mod_exp(E, b, p)   # 8^3 mod 23 = 512 mod 23 = 6
print(f"\n  Bob-Eve Key (K_EB):")
print(f"    Bob computes:   K_EB = E^b mod p = {E}^{b} mod {p} = {K_EB}")

# Eve computes the same Bob-Eve key:
# Eve knows her own secret (e=6) and Bob's real public key (B=10)
# Formula: K_EB = B^e mod p
eve_to_bob = mod_exp(B, e, p)   # 10^6 mod 23 = 512 mod 23 = 6
print(f"    Eve computes:   K_EB = B^e mod p = {B}^{e} mod {p} = {eve_to_bob}")

# ── Step (d): Impact Summary ──
print(f"\n[Step d] Attack Result:")
print(f"  Alice <──── K_AE = {K_AE} ────> Eve")
print(f"  Eve   <──── K_EB = {K_EB} ────> Bob")
print(f"")
print(f"  Eve can now:")
print(f"    1. Receive Alice's message (encrypted with K_AE={K_AE})")
print(f"    2. Decrypt it using K_AE={K_AE}")
print(f"    3. Read the plaintext")
print(f"    4. Re-encrypt it using K_EB={K_EB}")
print(f"    5. Forward it to Bob")
print(f"")
print(f"  Neither Alice nor Bob detect anything wrong,")
print(f"  because standard DH provides NO authentication.")


  SECTION 2: Diffie-Hellman Key Exchange

[Public Parameters]
  Prime modulus  p = 23
  Generator      g = 5

[Private Secrets]
  Alice's secret a = 4
  Bob's secret   b = 3

[Step a] Alice computes her public value:
  A = g^a mod p = 5^4 mod 23 = 4

[Step b] Bob computes his public value:
  B = g^b mod p = 5^3 mod 23 = 10

[Step c] Alice computes the shared secret:
  s = B^a mod p = 10^4 mod 23 = 18

[Step d] Bob computes the shared secret:
  s = A^b mod p = 4^3 mod 23 = 18

[Step e] Verification:
  ✓ SUCCESS! Both computed the same shared secret: 18


  SECTION 3: Man-in-the-Middle Attack (Eve)

[Setup] Eve's private key: e = 6
  Eve's public value: E = g^e mod p = 5^6 mod 23 = 8

[Step a] Alice sends A=4 to Bob.
  Eve INTERCEPTS A=4 and sends her own value E=8 to Bob instead.
  Bob now thinks he received Alice's key, but it's Eve's.

[Step b] Bob sends B=10 to Alice.
  Eve INTERCEPTS B=10 and sends her own value E=8 to Alice instead.
  Alice now thinks she received Bob's key, but it